In [1]:
import requests
import re
import nltk
from nltk.corpus import stopwords
import html
import unicodedata
from typing import List
from xml.etree import ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import re
import json
import time
from tqdm import tqdm
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv()

/Users/justpqa/advpai/venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


True

In [2]:
advp1 = pd.read_csv("test_tables/ADVP_1026_v3p8_extracted.txt", sep = "\t", encoding="cp1252")
advp1_all_pmid = advp1["Pubmed ID"].apply(lambda x: str(int(x)) if not pd.isna(x) else "").unique()
advp1_all_pmid = [pmid for pmid in advp1_all_pmid if pmid != ""]

In [3]:
advp2 = pd.read_excel("new_gwas_ad_paper_only_tiab_extended.xlsx")
advp2_all_pmid = advp2["pmid"].unique()
advp2_all_pmid = [pmid for pmid in advp2_all_pmid if pmid != ""]

In [4]:
advp2_1 = pd.read_excel("new_gwas_ad_paper_only_tiab_extended_1.xlsx")
advp2_1_all_pmid = advp2_1["pmid"].unique()
advp2_1_all_pmid = [pmid for pmid in advp2_1_all_pmid if pmid != ""]

In [6]:
def fetch_titles_abstracts_batch(batch_pmids):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    results = {}
    batch_size = 50
    for i in range(0, len(batch_pmids), batch_size):
        pmid_str = ",".join(map(str, batch_pmids[i: min(i + batch_size, len(batch_pmids))]))

        params = {
            "db": "pubmed",
            "id": pmid_str,
            "retmode": "xml",
            "api_key": os.environ.get("ENTREZ_API_KEY", "")
        }

        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()

        root = ET.fromstring(r.content)

        for article in root.findall(".//PubmedArticle"):
            pmid_elem = article.find("./MedlineCitation/PMID")
            title_elem = article.find(".//ArticleTitle")
            abstract_elems = article.findall(".//Abstract/AbstractText")

            if pmid_elem is None or pmid_elem.text is None:
                continue

            pmid = pmid_elem.text

            # Extract title
            if title_elem is not None:
                title = "".join(title_elem.itertext()).strip()
            else:
                title = ""

            # Extract abstract (can have multiple sections)
            abstract_parts = []
            for ab in abstract_elems:
                text = "".join(ab.itertext()).strip()
                if text:
                    label = ab.attrib.get("Label")
                    if label:
                        abstract_parts.append(f"{label}: {text}")
                    else:
                        abstract_parts.append(text)

            abstract = " ".join(abstract_parts)

            # Combine into a single paragraph
            if title and abstract:
                combined = f"{title} {abstract}"
            elif title:
                combined = title
            elif abstract:
                combined = abstract
            else:
                combined = None

            results[pmid] = combined

    return results

In [7]:
# need to clean text for better count
def clean_text(text):
    if not text:
        return text

    # Decode HTML entities (&amp;, &#x2013;, etc.)
    text = html.unescape(text)

    # Normalize unicode (e.g., fancy quotes → standard)
    text = unicodedata.normalize("NFKC", text)

    # Remove weird control characters
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# also remove stop word
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
def remove_stopwords(text):
    # Keep hyphenated words like genome-wide, case-control, TNF-alpha
    words = re.findall(r"\b\w+(?:-\w+)*\b", text.lower())
    filtered = [w for w in words if w not in stop_words]
    return " ".join(filtered)


# combine
def clean_text_and_remove_stopwords(text):
    text = clean_text(text)
    text = remove_stopwords(text)
    return text

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/justpqa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
pmid_to_tiab = fetch_titles_abstracts_batch(advp1_all_pmid)
max_n_grams = 4
counter = {}
for pmid in pmid_to_tiab:
    appear_in_paper = set()
    word_lst = clean_text_and_remove_stopwords(pmid_to_tiab[pmid]).lower().split(" ")
    for n in range(1, max_n_grams + 1):
        for i in range(len(word_lst) - n + 1):
            term = " ".join(word_lst[i: i+n])
            appear_in_paper.add(term)
    for term in appear_in_paper:
        counter[term] = counter.get(term, 0) + 1
temp = {
    "Term": [],
    "Count appearances ADVP1": []
}
for term in counter:
    temp["Term"].append(term)
    temp["Count appearances ADVP1"].append(counter[term])
tiab_terms_counter_df = pd.DataFrame(temp)
tiab_terms_counter_df = tiab_terms_counter_df.sort_values("Count appearances ADVP1", ascending = False).reset_index().drop("index", axis = 1)
tiab_terms_counter_df.to_csv("tiab_terms_counter_df_1.csv", index = False)

In [9]:
pmid_to_tiab = fetch_titles_abstracts_batch(advp2_all_pmid)
max_n_grams = 4
counter = {}
for pmid in pmid_to_tiab:
    appear_in_paper = set()
    word_lst = clean_text_and_remove_stopwords(pmid_to_tiab[pmid]).lower().split(" ")
    for n in range(1, max_n_grams + 1):
        for i in range(len(word_lst) - n + 1):
            term = " ".join(word_lst[i: i+n])
            appear_in_paper.add(term)
    for term in appear_in_paper:
        counter[term] = counter.get(term, 0) + 1
temp = {
    "Term": [],
    "Count appearances ADVP2": []
}
for term in counter:
    temp["Term"].append(term)
    temp["Count appearances ADVP2"].append(counter[term])
tiab_terms_counter_df = pd.DataFrame(temp)
tiab_terms_counter_df = tiab_terms_counter_df.sort_values("Count appearances ADVP2", ascending = False).reset_index().drop("index", axis = 1)
tiab_terms_counter_df.to_csv("tiab_terms_counter_df_2.csv", index = False)

In [10]:
pmid_to_tiab = fetch_titles_abstracts_batch(advp2_1_all_pmid)
max_n_grams = 4
counter = {}
for pmid in pmid_to_tiab:
    appear_in_paper = set()
    word_lst = clean_text_and_remove_stopwords(pmid_to_tiab[pmid]).lower().split(" ")
    for n in range(1, max_n_grams + 1):
        for i in range(len(word_lst) - n + 1):
            term = " ".join(word_lst[i: i+n])
            appear_in_paper.add(term)
    for term in appear_in_paper:
        counter[term] = counter.get(term, 0) + 1
temp = {
    "Term": [],
    "Count appearances ADVP2.1": []
}
for term in counter:
    temp["Term"].append(term)
    temp["Count appearances ADVP2.1"].append(counter[term])
tiab_terms_counter_df = pd.DataFrame(temp)
tiab_terms_counter_df = tiab_terms_counter_df.sort_values("Count appearances ADVP2.1", ascending = False).reset_index().drop("index", axis = 1)
tiab_terms_counter_df.to_csv("tiab_terms_counter_df_2_1.csv", index = False)